In [1]:
!pip install speechbrain

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 864.1/864.1 kB 14.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.7/117.7 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.2/722.2 kB 30.7 MB/s eta 0:00:00


In [2]:
!pip install torchmetrics[audio]

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 97.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.1 MB/s eta 0:00:00
  Created wheel for pesq: filename=pesq-0.0.4-cp310-cp310-linux_x86_64.whl size=262944 sha256=5852a61ce780f80ca0d79f7a0068d691fd59356b60d04079017cfed84c8c9784
  Stored in directory: /root/.cache/pip/wheels/c5/4e/2c/251524370c0fdd659e99639a0fbd0ca5a782c3aafcd456b28d
Successfully built pesq


In [3]:
!pip install https://github.com/ludlows/python-pesq/archive/master.zip

     - 223.8 kB 4.7 MB/s 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pesq: filename=pesq-0.0.5-cp310-cp310-linux_x86_64.whl size=262409 sha256=9d9c49e239d9c516fc18d364a644a4795787bd2845b18ec20e219ddfdfa43bf6
  Stored in directory: /tmp/pip-ephem-wheel-cache-vagni_dk/wheels/3d/7d/9f/9e1d63dc212910a515ac320c8ebfa8467839b0fef3fc1bb57f
Successfully built pesq
  Attempting uninstall: pesq
    Found existing installation: pesq 0.0.4
    Uninstalling pesq-0.0.4:
      Successfully uninstalled pesq-0.0.4


In [4]:
import pandas as pd
import librosa

from torchmetrics.audio import SignalDistortionRatio
from speechbrain.inference.separation import SepformerSeparation as separator
import numpy as np
import torchaudio
import torchaudio.transforms as T

In [5]:
from torchmetrics.audio.pesq import PerceptualEvaluationSpeechQuality

In [6]:
import torch
print(torch.cuda.is_available())

True


In [7]:
SAMPLE_RATE = 8000

In [8]:
def get_permutations():
    perms = [
        (sdr(x1, pred1) + sdr(x2, pred2)) / 2,
        (sdr(x1, pred2) + sdr(x2, pred1)) / 2
    ]
    best_score = max(perms)

sdr = SignalDistortionRatio()
def calculate_sdr(preds, target):
    return sdr(preds, target)

wb_pesq = PerceptualEvaluationSpeechQuality(SAMPLE_RATE, 'nb')
def calculate_wb_pesq(preds, target):
    return wb_pesq(preds, target)

def read_file(file_path: str, new_sample_rate=None):
    waveform, sample_rate = torchaudio.load(file_path)
    if new_sample_rate:
        resampler = T.Resample(sample_rate, new_sample_rate, dtype=waveform.dtype)
        if sample_rate != new_sample_rate:
            waveform = resampler(waveform)
    return waveform
    

In [9]:
model = separator.from_hparams(source="speechbrain/resepformer-wsj02mix", savedir='pretrained_models/resepformer-wsj02mix', run_opts={"device": "cuda"})

hyperparams.yaml:   0%|          | 0.00/1.51k [00:00<?, ?B/s]

encoder.ckpt:   0%|          | 0.00/9.07k [00:00<?, ?B/s]

masknet.ckpt:   0%|          | 0.00/186M [00:00<?, ?B/s]

decoder.ckpt:   0%|          | 0.00/9.00k [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/speechbrain/utils/checkpoints.py:200: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(path, map_location=device

In [10]:
dataset_path = "/kaggle/input/voice-dataset/combinations/combinations"

In [11]:
from pathlib import Path
from tqdm import tqdm
import time

def process_dataset(df, model, model_name, model_sample_rate=8000, out_dir='output', dataset_path=dataset_path):
    input_path = Path(dataset_path)
    
    out1_dir = Path(out_dir + "_" + model_name) / "out1"
    out2_dir = Path(out_dir + "_" + model_name) / "out2"
    out1_dir.mkdir(parents=True, exist_ok=True)
    out2_dir.mkdir(parents=True, exist_ok=True)
    separation_times = []

    for idx, row in tqdm(df.iterrows()):
        mix_id = row['ID']
        filename = Path(f"combo_{str(mix_id)}.wav")
        input_wave = read_file(input_path / filename)

        st = time.time()
        est_sources = model.separate_batch(input_wave)  # [2, T] — размерность выхода
        separation_times.append(time.time() - st)

        torchaudio.save(str(out1_dir / f"{filename}"), est_sources[:, :, 0].detach().cpu(), model_sample_rate)
        torchaudio.save(str(out2_dir / f"{filename}"), est_sources[:, :, 1].detach().cpu(), model_sample_rate)
    return separation_times

In [12]:
csv_path = "/kaggle/input/voice-dataset/audio_combinations.csv"

In [13]:
data = pd.read_csv(csv_path)

In [14]:
data.head()

,ID,genders,pattern,SNR,path1,path2,sentence1,sentence2,duration,overlap_ratio,start1,end1,start2,end2
0,1,М-М,Частичное перекрытие,-5,cv-corpus-21.0-2025-03-14/ru\clips\common_voic...,cv-corpus-21.0-2025-03-14/ru\clips\common_voic...,"Все ожидали, что это будет кринжовый сериал, н...","Третьим аспектом, которым нам следует заняться...",5.076,1.0,0.0,5.076,2.538,7.674
1,2,М-М,Частичное перекрытие,0,cv-corpus-21.0-2025-03-14/ru\clips\common_voic...,cv-corpus-21.0-2025-03-14/ru\clips\common_voic...,Мир добился замечательного прогресса в борьбе ...,"Однако, хотя ответственность за тупик, несомне...",5.016,1.0,0.0,5.016,2.508,10.956
2,3,М-М,Частичное перекрытие,5,cv-corpus-21.0-2025-03-14/ru\clips\common_voic...,cv-corpus-21.0-2025-03-14/ru\clips\common_voic...,да,Что она думает?,1.944,1.0,0.0,1.944,0.972,3.780
3,4,М-М,Параллельный диалог,-5,cv-corpus-21.0-2025-03-14/ru\clips\common_voic...,cv-corpus-21.0-2025-03-14/ru\clips\common_voic...,"Кстати, в рамках данного заседания времени у н...",Сейчас слово предоставляется представителю Кан...,5.544,1.0,0.0,5.544,0.000,4.176
4,5,М-М,Параллельный диалог,0,cv-corpus-21.0-2025-03-14/ru\clips\common_voic...,cv-corpus-21.0-2025-03-14/ru\clips\common_voic...,"Во-вторых, принадлежащие членам кооперативы — ...","Нет таких учебников иврита, которые бы могли з...",9.144,1.0,0.0,9.144,0.000,6.408


In [ ]:
find_dir_dict = {}
for dir_name in os.listdir(common_voice_dir):
    input_dir = os.listdir(os.path.join(common_voice_dir, dir_name))
    find_dir_dict[input_dir[0]] = dir_name

In [78]:
data["path1"].iloc[0]

'cv-corpus-21.0-2025-03-14/ru\\clips\\common_voice_ru_37264600.mp3'

In [85]:
data["path2"].iloc[0]

'cv-corpus-21.0-2025-03-14/ru\\clips\\common_voice_ru_22460934.mp3'

In [79]:
find_dir_dict["cv-corpus-21.0-2025-03-14"]

'14765216062 14765215733'

In [84]:
os.listdir("/kaggle/input/common-voice-rus/14765216062 14765215733//cv-corpus-21.0-2025-03-14/ru/clips")

['common_voice_ru_42544229.mp3']

In [70]:
common_voice_dir = "/kaggle/input/common-voice-rus"


{'cv-corpus-21.0-2025-03-14': '14765216062 14765215733'}

In [15]:
separation_times = process_dataset(data, model, "resepformer-wsj02mix")

90it [00:16,  5.43it/s]


90it [00:16,  5.54it/s]


In [73]:

def compute_file_metrics(row: dict, model_name: str, out_dir="output") -> tuple:
    input_path = Path(dataset_path)
    mix_id = row['ID']
    mix_file_path = input_path / Path(f"combo_{str(mix_id)}.wav")
    
    model_answer_1_path = Path(out_dir + "_" + model_name) / "out1" / mix_file_path
    model_answer_2_path = Path(out_dir + "_" + model_name) / "out2" / mix_file_path
    
    sample_time = row['duration'] / SAMPLE_RATE

    dir_name_1 = find_dir_dict[row["path1"].split("/")[0]]
    dir_name_2 = find_dir_dict[row["path2"].split("/")[0]]
    original_1 = read_file(os.path.join(common_voice_dir, dir_name_1, row["path1"]))
    original_2 = read_file(os.path.join(common_voice_dir, dir_name_2, row["path2"]))

    orig_duration_1 = original_1.size / SAMPLE_RATE
    orig_duration_2 = original_2.size / SAMPLE_RATE
    orig_duration_in_mix_1 = int(row['end1']) - int(row['start1'])
    orig_duration_in_mix_2 = int(row['end2']) - int(row['start2'])
    s1, e1 = orig_duration_in_mix_1 * sample_time
    s2, e2 = orig_duration_in_mix_2 * sample_time
    target1 = orig_duration_1[s1:e1]
    target2 = orig_duration_2[s2:e2]
    
    model_answer_1 = read_file(os.path.join(model_answer_1_path, row["path1"]))
    model_answer_2 = read_file(os.path.join(model_answer_2_path, row["path2"]))

    mix_waveform = read_file(mix_file_path)

    best_sdr = -np.inf
    best_pesq = -np.inf

    for answer_waveform_1, answer_waveform_2 in [(model_answer_1_path, model_answer_2_path),
                                                 (model_answer_2_path, model_answer_1_path)]:
        pesq_1 = calculate_wb_pesq(answer_waveform_1, target1)
        sdr_1 = calculate_sdr(answer_waveform_1, target1)
        
        pesq_2 = calculate_wb_pesq(answer_waveform_2, target2)
        sdr_2 = calculate_sdr(answer_waveform_2, target2)

        mean_pesq = (pesq_1 + pesq_2) / 2
        mean_sdr = (sdr_1 + sdr_2) / 2

        best_sdr = max(mean_sdr, best_sdr)
        best_pesq = max(mean_pesq, best_pesq)
    return best_sdr, best_pesq

In [74]:
row_example = data.iloc[0]

In [75]:
os.listdir("/kaggle/input/common-voice-rus/14765216062 14765215733/cv-corpus-21.0-2025-03-14/ru/clips\common_voice_ru_37264600")

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/common-voice-rus/14765216062 14765215733/cv-corpus-21.0-2025-03-14/ru/clips\\common_voice_ru_37264600'

In [76]:
compute_file_metrics(row_example, "resepformer-wsj02mix")

RuntimeError: Failed to open the input "/kaggle/input/common-voice-rus/14765216062 14765215733/cv-corpus-21.0-2025-03-14/ru\clips\common_voice_ru_37264600.mp3" (No such file or directory).
Exception raised from get_input_format_context at /__w/audio/audio/pytorch/audio/src/libtorio/ffmpeg/stream_reader/stream_reader.cpp:42 (most recent call first):
frame #0: c10::Error::Error(c10::SourceLocation, std::string) + 0x96 (0x7f3c3e149446 in /usr/local/lib/python3.10/dist-packages/torch/lib/libc10.so)
frame #1: c10::detail::torchCheckFail(char const*, char const*, unsigned int, std::string const&) + 0x64 (0x7f3c3e0f36e4 in /usr/local/lib/python3.10/dist-packages/torch/lib/libc10.so)
frame #2: <unknown function> + 0x42134 (0x7f3c07069134 in /usr/local/lib/python3.10/dist-packages/torio/lib/libtorio_ffmpeg4.so)
frame #3: torio::io::StreamingMediaDecoder::StreamingMediaDecoder(std::string const&, std::optional<std::string> const&, std::optional<std::map<std::string, std::string, std::less<std::s